# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle: Growth / Recovery / Momentum Prediction.**

I'm choosing this over the four predefined lanes because the starter dataset's own `trend_direction`
label is a snapshot bucket, not a forecast — and I want to build something that actually looks forward
instead of describing "what already happened." The client work I'm doing at SafeX (attack-surface
scoping, i.e. finding what's exposed *before* it's exploited) and the AI-agent-security reading I've
been doing both come down to the same shape of question: "given signals I can see now, what happens
next, and is it worth someone's attention?" This lane is the FlyRank version of that question, and it
is also the one predefined-vs-freestyle option that most directly requires a real prior-window →
future-window label, so it forces me to practice the leakage discipline (`hunting-leakage-and-validating`)
that the other lanes can mostly avoid.

Why not a core lane instead: Refresh/Content Opportunity Scoring and CTR Opportunity Scoring both score
a *current* state (a snapshot), which is closer to a ranking/EDA task than a forecasting task. I want the
harder, more failure-prone version of the problem on purpose, while it's still cheap to be wrong (Week 1,
not Week 7).

In [1]:
# Nothing to compute for the lane statement itself — the numbers that justify
# it live in Section 3. This cell is intentionally left running clean.
print("Lane: Growth / Recovery / Momentum Prediction (freestyle)")


Lane: Growth / Recovery / Momentum Prediction (freestyle)


## 2. The question: decision, action, cost of a wrong call

**Search question:** Given a page's search and engagement signals over a prior window, can we flag,
before the fact, which pages are about to decline, which are about to recover, and which are about to
gain momentum over the *next* window?

**Unit of analysis:** one content page (`content_id`), scored at a point in time (a decision date), using
only signals known up to that date. In the warehouse this becomes a `content_id x report_date` row where
I choose the decision date myself; in the starter CSV the closest proxy is the `prev_30d` window as
"prior" and the `last_30d` window as the closest thing to a partially-observed "next" window.

**Output:** a ranked list of pages, split into three future-facing buckets (likely decline / likely
recover / likely gain momentum next window), each with a score and a short reason code — not a single
"declining: yes/no" flag.

**Who acts on it, and what do they do:** a content/SEO account lead at FlyRank, deciding which pages to
put in front of a client this week for a refresh, and which recovering pages to *leave alone* so nobody
wastes a refresh on something that was already turning around on its own.

**Cost of a wrong call:**
- **False decline flag (predicted decline, page was actually fine):** wasted editor/writer hours on a
  refresh that wasn't needed, and a page that gets touched (and risks a temporary ranking dip from the
  edit itself) for no reason.
- **Missed decline (predicted fine, page actually kept sliding):** real traffic/revenue lost before
  anyone notices, and a client conversation that happens too late to look proactive.
- **False recovery flag (predicted recovering, actually still declining):** the team stands down on a
  page that needed help, compounding the loss.
- Because a missed decline is usually more expensive than a wasted review hour, I'd rather tune the
  model toward higher recall on "likely decline" even if precision drops a bit — but that's a threshold
  decision I'll revisit once I have real precision/recall numbers, not something to bake in on Day 1.

**Why data or ML help at all, instead of a plain rule:** a plain rule like "flag if last 30 days is below
prior 30 days" already exists in this data as `trend_pct`/`trend_direction` — and Section 3 shows that
rule is *already* flagging 54% of all pages as "down," which is far too blunt to hand a human a
prioritized list. Multiple signals (position, impressions, engagement, freshness, content age, client
history depth) move together in ways that aren't a single clean threshold — that's the "many signals,
tangled, shifting over time" condition where ML earns its place over an if-statement.

In [2]:
# Supporting check for Section 2's claim: the existing simple rule is too blunt to be an action list.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows flagged 'down' by the existing snapshot rule (trend_direction):",
      (df['trend_direction'] == 'down').sum(), "/", len(df),
      f"({(df['trend_direction'] == 'down').mean()*100:.1f}%)")
print()
print("If a review team can only look at, say, 200 pages a week, a 54% 'down' rate")
print("gives them no way to choose which 200 of ~16,000 flagged pages to start with.")


Rows flagged 'down' by the existing snapshot rule (trend_direction): 16262 / 30000 (54.2%)

If a review team can only look at, say, 200 pages a week, a 54% 'down' rate
gives them no way to choose which 200 of ~16,000 flagged pages to start with.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Starter dataset:", df.shape[0], "rows,", df.shape[1], "columns,",
      df['client_id'].nunique(), "clients")

# --- Number 1: the label FlyRank already tracks (trend_direction) is a snapshot bucket ---
print()
print("trend_direction breakdown:")
print(df['trend_direction'].value_counts())
print((df['trend_direction'].value_counts(normalize=True) * 100).round(1))

# --- Number 2: trend_pct is (almost) exactly last_30d vs prev_30d impressions % change ---
usable = df[df['impressions_prev_30d'] > 0].copy()
usable['momentum_pct'] = (
    (usable['impressions_last_30d'] - usable['impressions_prev_30d'])
    / usable['impressions_prev_30d'] * 100
)
corr = usable['trend_pct'].corr(usable['momentum_pct'])
print()
print(f"Correlation between trend_pct and (last_30d vs prev_30d) momentum: {corr:.6f}")
print("-> trend_direction is a relabeling of a CURRENT window comparison, not a forecast.")
print("   This is exactly the 'label trap' the flyrank-data skill warns about:")
print("   trend_direction/trend_pct describe what already happened, so I cannot use them")
print("   as features for a forward-looking model — only as a baseline to try to beat.")

# --- Number 3: rows where the recent 30 days are already diverging from the longer trend ---
fading = usable[(usable['trend_direction'].isin(['up', 'stable'])) & (usable['momentum_pct'] < 0)]
denom = len(usable[usable['trend_direction'].isin(['up', 'stable'])])
print()
print(f"Pages currently labeled 'up' or 'stable' whose most recent 30 days already")
print(f"trail their prior 30 days: {len(fading)} / {denom} = {len(fading)/denom*100:.1f}%")
print("-> a third of 'healthy' pages already show early fade inside the window most")
print("   people would call fine. That's a real momentum signal the single-bucket label throws away —")
print("   and it's the gap this lane is trying to catch earlier, with proper future-window labels")
print("   built from the warehouse's daily fact table rather than from a fixed prev/last split.")

# --- Bonus: client spread, relevant to grouped validation later ---
sizes = df.groupby('client_id').size()
print()
print(f"{df['client_id'].nunique()} clients, pages per client ranges from "
      f"{sizes.min()} to {sizes.max()} (median {int(sizes.median())}).")
print("-> wide spread means a random train/test split would leak client-level habits across")
print("   splits; any validation later needs to group by client_id.")


Starter dataset: 30000 rows, 44 columns, 32 clients

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Correlation between trend_pct and (last_30d vs prev_30d) momentum: 1.000000
-> trend_direction is a relabeling of a CURRENT window comparison, not a forecast.
   This is exactly the 'label trap' the flyrank-data skill warns about:
   trend_direction/trend_pct describe what already happened, so I cannot use them
   as features for a forward-looking model — only as a baseline to try to beat.

Pages currently labeled 'up' or 'stable' whose most recent 30 days already
trail their prior 30 days: 3454 / 10350 = 33.4%
-> a third of 'healthy' pages already show early fade inside the window most
   people would call fine. That's a real momentum signal the single-bucket labe

## 4. Careful words: what I can and can't claim

**What I can claim, once this is built:**
- **Observed:** rates of decline/recovery/growth as measured by the actual signals in the data
  (impressions, clicks, sessions, position, engagement, scroll), over windows I define and document.
- **Directional / decision-support:** a ranked list saying "these pages are relatively more likely to
  decline (or recover, or gain) next window than those pages," with a stated precision@K on held-out,
  client-grouped data — not a guaranteed individual outcome for any one page.
- **Reproducibility of a known pattern:** if I ever try to rebuild `trend_direction` itself, I will say
  plainly "this reproduces an existing rule" and not present it as a discovery.

**What I will never claim:**
- **Not causal.** I cannot say a refresh *caused* recovery unless I run an actual before/after
  experiment with a comparison group — correlation between "was refreshed" and "later recovered" is not
  proof, since the pages picked for refresh were probably not picked at random.
- **Not "predicting Google."** I have no access to ranking algorithm internals, so I'm modeling
  *observable outcomes* (impressions, clicks, position as reported), never claiming to explain or predict
  a search engine's internal ranking factors.
- **Not AI-citation or AI-ranking claims.** `ai_sessions_90d` only measures sessions with a click-through
  from an AI tool — not whether an AI system "cited" or "ranked" the content.
- **Not a client-name- or query-level claim.** IDs are pseudonyms; I will only ever talk about aggregate
  patterns, never try to guess which real client or query a row corresponds to.
- **Not overclaiming skill.** Any accuracy number has to be checked against a naive baseline (e.g. "always
  predict the majority class" or the existing `trend_direction` rule) and against seasonality — a model
  that just rides a seasonal wave everyone's data shares is not "model skill," it's the calendar.

In [4]:
# Sanity check that supports Section 4's "not model skill" caution:
# even the crude existing rule already labels the MAJORITY class as 'down',
# so any model I build has to clear this bar, not just beat 50/50 guessing.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
majority_share = df['trend_direction'].value_counts(normalize=True).max()
print(f"Naive 'always predict the majority label' baseline would already be right "
      f"{majority_share*100:.1f}% of the time on trend_direction.")
print("Any model I build later needs to be judged against this number, and against")
print("precision@K on a client-grouped, time-aware split -- not against a bare accuracy score.")


Naive 'always predict the majority label' baseline would already be right 54.2% of the time on trend_direction.
Any model I build later needs to be judged against this number, and against
precision@K on a client-grouped, time-aware split -- not against a bare accuracy score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.